# DeepAR Deep Learning Forecasting

Split from the original combined demo (Task 1). Trains a **DeepAR** probabilistic
forecasting model (PyTorch Forecasting + Lightning) on Walmart weekly sales.

**Compute:** run on **Serverless GPU (AI Runtime)** — select an **A10** accelerator in the
Environment panel (right sidebar) before running. Cell 2 confirms `torch.cuda.is_available()`.

**Source table:** `shm_skunkworks_catalog.forecasting_brickfood.walmart_training`

In [ ]:
# Install DL deps first (GPU node has CUDA; torch wheel is CUDA-enabled)
%pip install --quiet torch pytorch-lightning pytorch-forecasting pytorch-tabular
dbutils.library.restartPython()

## Setup — self-contained data generation
Installs deps, then creates the source table (idempotent).

In [ ]:
# --- Setup: generate the source data (self-contained; idempotent) ---
spark.sql("CREATE SCHEMA IF NOT EXISTS shm_skunkworks_catalog.forecasting_brickfood")
spark.sql("""
CREATE TABLE IF NOT EXISTS shm_skunkworks_catalog.forecasting_brickfood.walmart_training AS
SELECT s.store AS store, d.dept AS dept,
       date_add(DATE'2022-01-07', 7*w.wk) AS `Date`,
       round(20000 + 6000*sin(2*pi()*(w.wk%52)/52.0) + 150*w.wk
             + s.store*1200 + d.dept*400 + (rand()*3000-1500), 2) AS `Weekly_Sales`
FROM (SELECT explode(sequence(0,149)) AS wk) w
CROSS JOIN (SELECT explode(array(1,2,3)) AS store) s
CROSS JOIN (SELECT explode(array(1,2,3)) AS dept) d
""")
print("walmart_training ready:",
      spark.table("shm_skunkworks_catalog.forecasting_brickfood.walmart_training").count(), "rows")

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
# code from: https://medium.com/@mnitin3/pytorch-forecasting-introduction-to-time-series-forecasting-706cbc48768

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

In [ ]:
%sql
select * from shm_skunkworks_catalog.forecasting_brickfood.walmart_training limit 10;

In [ ]:
df = spark.read.table("shm_skunkworks_catalog.forecasting_brickfood.walmart_training").filter("Store = 1 AND Dept = 1").toPandas()
df = df.drop(columns=["store", "dept"])
df["series"] = "dummy"
display(df)

In [ ]:
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline, NaNLabelEncoder
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
import torch
from sklearn.model_selection import train_test_split

# Load data (your data is already in DataFrame `df`)
df["Date"] = pd.to_datetime(df["Date"])

# Create time_idx (needed by PyTorch Forecasting)
df = df.sort_values("Date")
df["time_idx"] = ((df["Date"] - df["Date"].min()) / pd.Timedelta(weeks=1)).astype(int)
display(df["time_idx"])

# Display data types of df time_idx
display(df["time_idx"].dtypes)

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting import Baseline
from pytorch_forecasting.metrics import RMSE
import pytorch_lightning as pl

from torch.utils.data import DataLoader

max_encoder_length = 10
validation_cutoff = 30
max_prediction_length = 3

# Ensure time_idx is of integer type
df["time_idx"] = df["time_idx"].astype(int)

training_cutoff = df["time_idx"].max() - max_prediction_length - validation_cutoff
print(training_cutoff)

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["series"],
    min_encoder_length=5,  # Can be less than max_encoder_length
    min_prediction_length=2,  # Can be less than max_prediction_length
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_unknown_reals=["Weekly_Sales"],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

validation =  TimeSeriesDataSet(
    df[(df.time_idx > training_cutoff) & (df.time_idx <= df["time_idx"].max() - max_prediction_length)],
    time_idx="time_idx",
    target="Weekly_Sales",
    group_ids=["series"],
    min_encoder_length=5,  # Can be less than max_encoder_length
    min_prediction_length=2,  # Can be less than max_prediction_length
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_unknown_reals=["Weekly_Sales"],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

print(type(training))
print(type(validation))
print(training.group_ids)
print(validation.group_ids)

In [ ]:
import warnings

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
import pandas as pd
from pandas.errors import SettingWithCopyWarning
import torch

from pytorch_forecasting import GroupNormalizer, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.examples import generate_ar_data
from pytorch_forecasting.metrics import NormalDistributionLoss
from pytorch_forecasting.models.deepar import DeepAR

warnings.simplefilter("error", category=SettingWithCopyWarning)

batch_size = 64
train_dataloader = training.to_dataloader(
    train=True, batch_size=batch_size, num_workers=0
)
val_dataloader = validation.to_dataloader(
    train=False, batch_size=batch_size, num_workers=0
)

print(train_dataloader.dataset.group_ids)
print(val_dataloader.dataset.group_ids)

# save datasets
training.save("training.pkl")
validation.save("validation.pkl")

early_stop_callback = EarlyStopping(
    monitor="val_loss", min_delta=1e-4, patience=5, verbose=False, mode="min"
)
lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices="auto",
    gradient_clip_val=0.1,
    limit_train_batches=30,
    limit_val_batches=3,
    # fast_dev_run=True,
    # logger=logger,
    # profiler=True,
    callbacks=[lr_logger, early_stop_callback],
)

deepar = DeepAR.from_dataset(
    training,
    learning_rate=0.1,
    hidden_size=32,
    dropout=0.1,
    loss=NormalDistributionLoss(),
    log_interval=10,
    log_val_interval=3,
    # reduce_on_plateau_patience=3,
)
print(f"Number of parameters in network: {deepar.size() / 1e3:.1f}k")

torch.set_num_threads(10)
trainer.fit(
    deepar,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

# calc mean absolute error on validation set
device = deepar.device
print(next(deepar.parameters()).device)
print(device)
actuals = torch.cat([y.to(device) for x, (y, weight) in iter(val_dataloader)])
actuals = actuals.to("cuda")
print(actuals)
predictions = deepar.predict(val_dataloader).to("cuda:0")
print(f"Mean absolute error of model: {(actuals - predictions).abs().mean()}")

# plot actual vs. predictions
preds = deepar.predict(val_dataloader, mode="raw", return_x=True)
# newer pytorch-forecasting returns a Prediction namedtuple
raw_predictions, x = preds.output, preds.x
try:
    for idx in range(3):
        deepar.plot_prediction(x, raw_predictions, idx=idx, add_loss_to_title=True)
except Exception as e:
    print('plot skipped:', e)

In [ ]:
print(actuals)
print(predictions)

In [ ]:
%sh

nvidia-smi

In [ ]:
import mlflow
import mlflow.pytorch

with mlflow.start_run() as run:
    mlflow.pytorch.log_model(deepar, artifact_path="pytorch-model")
    result = mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/pytorch-model",
        name="deepar_walmart_forecast",
    )
    print(f"Registered model version: {result.version}")